In [13]:
import requests
import pandas as pd
from time import sleep

In [2]:
BASE = "https://pheweb.org/UKB-TOPMed"

In [12]:
variants = {'rs10756069' :'9-10623666-G-A',
'rs1040910' :'9-10637778-C-T',
'rs10756068' :'9-10620421-T-A',
'rs10809140' :'9-10625637-A-T',
'rs10809143' :'9-10646045-T-G',
'rs2208707' :'9-10649660-G-A',
'rs1322302' :'9-10644978-C-A',
'rs116850462' :'11-131857962-C-T',
'rs35866887' :'2-2764330-AG-A',
'rs2045780' :'2-20125958-T-C',
'rs16904668' :'8-132347604-T-C',
'rs61695510' :'6-43167202-T-C',
'rs4853772' :'2-2764809-G-A',
'rs4853873' :'2-2763730-G-A',
'rs78153233' :'11-134509403-C-T',
'rs117129241' :'11-134510086-C-T',
'rs114321875' :'11-134510156-T-C',
'rs55976342' :'6-43167905-T-C',
'rs60733214' :'6-43167996-G-A'}

In [14]:
def get_variant(variant):
    url = f"{BASE}/api/variant/{variant}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

In [15]:
results = {}

for rs, var in variants.items():
    data = get_variant(var)
    df = pd.DataFrame(data['phenos'])
    results[rs] = df
    sleep(0.5)


In [17]:

# Stack everything, tagging each row with its rsID
combined = pd.concat(
    [df.assign(rsid=rs) for rs, df in results.items()],
    ignore_index=True,
)
# Put rsid first
combined = combined[["rsid"] + [c for c in combined.columns if c != "rsid"]]

with pd.ExcelWriter("variants_phewas.xlsx", engine="openpyxl") as writer:
    combined.to_excel(writer, sheet_name="All", index=False)
    for rs, df in results.items():
        df.sort_values('pval').to_excel(writer, sheet_name=rs[:31], index=False)  # Excel caps sheet names at 31 chars
